# Лабораторная работа №3
## Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.

### Выполнил студент группы РТ5-61Б Иванов А.А.

**Glass Classification**

https://www.kaggle.com/datasets/uciml/glass/data

Описание:

Это набор идентификационных данных Glass от UCI. Он содержит 10 атрибутов, включая id. Ответом будет тип стекла (дискретные 7 значений)

Информация об атрибуте содержимого:

ID: от 1 до 214 (удален из CSV-файла)
RI: показатель преломления 
Na: натрий (единица измерения: массовая доля в соответствующем оксиде, атрибуты 4-10)
Mg: Магний
Al: Алюминий 
Si: Кремний
K: Калий
Ca: Кальций
Ba: Барий
Fe: Железо
Тип стекла: (атрибут класса) 
- 1 building_windows_float_processed
- 2 building_windows_non_float_processed
- 3 vehicle_windows_float_processed
- 4 vehicle_windows_non_float_processed (none in this database)
- 5 containers
- 6 tableware
- 7 headlamps

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
data = pd.read_csv("glass.csv", sep=",")

In [3]:
data["Type"].value_counts()

Type
2    76
1    70
7    29
3    17
5    13
6     9
Name: count, dtype: int64

In [4]:
X = data.drop('Type', axis=1)
y = data['Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [5]:
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

In [6]:
def collect_metrics(y_true, y_pred, average='weighted'):
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average=average),
        'Recall': recall_score(y_true, y_pred, average=average),
        'F1-Score': f1_score(y_true, y_pred, average=average),
    }

    return metrics

In [7]:
pipe_knn.fit(X_train, y_train)

y_test_pred = pipe_knn.predict(X_test)

metrics = {
        'test': collect_metrics(y_test, y_test_pred)
    }

print(metrics)

{'test': {'Accuracy': 0.7209302325581395, 'Precision': 0.6806201550387597, 'Recall': 0.7209302325581395, 'F1-Score': 0.6944910208743408}}


/home/alexandr/projects/TML/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 13],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski']
}

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipe_knn, 
    param_grid, 
    cv=stratified_cv, 
    scoring='f1_weighted',
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_test_pred_best=best_model.predict(X_test)


print(collect_metrics(y_test, y_test_pred))
print(collect_metrics(y_test, y_test_pred_best))

Best params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Best CV score: 0.6672
{'Accuracy': 0.7209302325581395, 'Precision': 0.6806201550387597, 'Recall': 0.7209302325581395, 'F1-Score': 0.6944910208743408}
{'Accuracy': 0.7906976744186046, 'Precision': 0.7930232558139534, 'Recall': 0.7906976744186046, 'F1-Score': 0.787796165342276}


/home/alexandr/projects/TML/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
